# 🦴 骨细胞分割 - 超参数自动优化 (HPO) 系统
# Osteoblast Segmentation with Hyperparameter Optimization

---

## 📋 工作流程概述

这是一个**完整的自检系统**，能够自动搜索最佳的超参数组合，而不是手动调参！

### 🔄 完整流程：

1. **QuPath 数据准备** → 导出图像块（Patches）到 Google Drive
2. **环境设置** → 安装依赖、连接 Google Drive、登录 W&B
3. **配置 HPO 搜索空间** → 定义要搜索的超参数范围（而非单一值）
4. **启动自动优化** → W&B Sweeps 自动运行 N 次实验
5. **分析结果** → 查看 W&B 仪表板，找到最佳配置
6. **使用最佳模型推理** → 对完整 WSI 进行预测
7. **QuPath 验收** → 导入预测结果进行验证

---

## ⚡ 什么是「自检系统」(HPO)？

### 旧方式（手动调参）：
- 您手动设置 `学习率=1e-4`，运行一次 → `Dice=0.80`
- 您手动改成 `学习率=5e-5`，再运行一次 → `Dice=0.82`
- ...重复数十次，既慢又繁琐 😓

### 新方式（自动 HPO）：
- 您设置**搜索范围**：`学习率: 1e-5 到 1e-3`，`批次大小: [8, 16]`
- 系统**自动运行 50 次实验**，每次使用不同的参数组合
- 系统会「根据结果调整」：如果发现低学习率效果好，它会集中在该范围搜索
- **最终给您报告**：「最佳组合是 `lr=3e-5, bs=16, U-Net`，Dice=0.85」🎉

---

## 🎯 开始之前的准备

### 必需条件：
1. ✅ QuPath 导出的图像块（images 和 labels 文件夹）
2. ✅ Google Drive 账号（存储数据和模型）
3. ✅ Weights & Biases (W&B) 免费账号 → [wandb.ai](https://wandb.ai)
4. ✅ Google Colab（推荐 Pro 以获得更好的 GPU）

---

**作者：** FTP Project  
**更新日期：** 2025-11-18  
**版本：** 1.0.0


---
# 📦 阶段一：环境设置
## Stage 1: Environment Setup
---

### 1.1 检查 GPU 并安装依赖

In [ ]:
# 检查 GPU 可用性
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU 可用: {gpu_name}")
    print(f"   显存: {gpu_memory:.2f} GB")
else:
    print("⚠️  未检测到 GPU，将使用 CPU（速度会很慢）")
    print("   建议：菜单栏 → 代码执行程序 → 更改运行时类型 → GPU")

In [ ]:
%%bash
# 安装核心依赖
pip install -q torch torchvision
pip install -q monai segmentation-models-pytorch
pip install -q albumentations opencv-python-headless
pip install -q wandb  # 超参数优化的核心库
pip install -q openslide-python  # WSI 处理
pip install -q pyyaml tqdm colorama

echo "✅ 所有依赖安装完成！"

### 1.2 连接 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("\n✅ Google Drive 已挂载到 /content/drive")
print("   您的文件路径示例: /content/drive/MyDrive/YourFolder")

### 1.3 登录 Weights & Biases

**首次使用？**
1. 访问 [wandb.ai](https://wandb.ai) 注册免费账号
2. 在 [设置页面](https://wandb.ai/authorize) 获取 API Key
3. 粘贴到下方

In [ ]:
import wandb

# 登录 W&B（会弹出输入框）
wandb.login()

print("\n✅ W&B 登录成功！")
print("   您现在可以使用超参数优化功能了")

### 1.4 克隆代码库（或上传项目文件）

In [ ]:
# 选项1: 从 GitHub 克隆（如果代码已上传）
# !git clone https://github.com/Zhaokun91/FTP.git
# %cd FTP

# 选项2: 从 Google Drive 加载（推荐用于个人项目）
import sys
sys.path.append('/content/drive/MyDrive/FTP')  # 替换为您的项目路径

print("✅ 代码库已加载")

---
# 📁 阶段二：数据结构说明
## Stage 2: Data Structure
---

### 预期的数据结构：

```
/content/drive/MyDrive/Patches/
├── images/              # QuPath 导出的图像块
│   ├── patch_001.png
│   ├── patch_002.png
│   └── ...
└── labels/              # QuPath 导出的标注掩码
    ├── patch_001.png    # 对应 images/patch_001.png
    ├── patch_002.png
    └── ...
```

**注意事项：**
- ✅ 图像和掩码文件名必须一一对应
- ✅ 掩码应为二值图像（0=背景，255=骨细胞）
- ✅ 支持格式：PNG、JPG、TIFF

In [ ]:
# 验证数据结构
from pathlib import Path

# 🔧 修改为您的数据路径
DATA_DIR = "/content/drive/MyDrive/Patches"  #@param {type:"string"}

data_path = Path(DATA_DIR)
image_dir = data_path / "images"
label_dir = data_path / "labels"

if not data_path.exists():
    print(f"❌ 数据目录不存在: {DATA_DIR}")
    print("   请检查路径是否正确")
else:
    num_images = len(list(image_dir.glob("*.*")))
    num_labels = len(list(label_dir.glob("*.*")))
    
    print(f"✅ 数据目录: {DATA_DIR}")
    print(f"   图像数量: {num_images}")
    print(f"   标注数量: {num_labels}")
    
    if num_images != num_labels:
        print(f"\n⚠️  警告: 图像和标注数量不匹配！")

---
# ⚙️ 阶段三：配置 HPO 搜索空间
## Stage 3: Configure HPO Search Space
---

### 🎯 这是「自检系统」的核心！

**关键区别：**
- ❌ 旧方式：您设置 `学习率 = 1e-4` （单一值）
- ✅ 新方式：您设置 `学习率: 1e-5 到 1e-3` （搜索范围）

系统会在这个范围内**智能搜索**，找到最佳值！

In [ ]:
# ===== A. 固定参数（不参与搜索）=====

# 数据路径
DATA_DIR = "/content/drive/MyDrive/Patches"  #@param {type:"string"}
OUTPUT_DIR = "/content/drive/MyDrive/HPO_Results"  #@param {type:"string"}

# W&B 项目名称
PROJECT_NAME = "Osteoblast_HPO"  #@param {type:"string"}

# 训练参数
NUM_EPOCHS = 30  #@param {type:"integer"}
NUM_WORKERS = 2  #@param {type:"integer"}
IMAGE_SIZE = 256  #@param {type:"integer"}
VAL_SPLIT = 0.2  #@param {type:"number"}
RANDOM_SEED = 42  #@param {type:"integer"}

# HPO 运行次数（越多越准确，但时间越长）
HPO_COUNT = 20  #@param {type:"integer"}

print("✅ 固定参数已设置")

In [ ]:
# ===== B. HPO 搜索空间（系统会自动尝试这些组合）=====

sweep_config = {
    'method': 'bayes',  # 贝叶斯优化（智能搜索）
    
    'metric': {
        'name': 'validation_dice',  # 优化目标：验证集 Dice 分数
        'goal': 'maximize'          # 越大越好
    },
    
    'parameters': {
        # 🔍 搜索参数 1: 模型架构
        'model_name': {
            'values': ['U-Net', 'U-Net++', 'FPN']  # 尝试这 3 种模型
        },
        
        # 🔍 搜索参数 2: 编码器（骨干网络）
        'encoder_name': {
            'values': ['resnet34', 'resnet50', 'efficientnet-b0']
        },
        
        # 🔍 搜索参数 3: 学习率（对数空间搜索）
        'learning_rate': {
            'distribution': 'log_uniform_values',
            'min': 1e-5,  # 最小值
            'max': 1e-3   # 最大值
        },
        
        # 🔍 搜索参数 4: 批次大小
        'batch_size': {
            'values': [8, 16]
        },
        
        # 🔍 搜索参数 5: 损失函数
        'loss_function': {
            'values': ['DiceLoss', 'FocalLoss', 'DiceFocalLoss']
        },
        
        # 🔍 搜索参数 6: 优化器
        'optimizer': {
            'values': ['Adam', 'AdamW']
        },
        
        # 🔍 搜索参数 7: 数据增强强度
        'augmentation_prob': {
            'distribution': 'uniform',
            'min': 0.3,
            'max': 0.7
        },
        
        # === 固定参数（传递给训练函数）===
        'num_epochs': {'value': NUM_EPOCHS},
        'num_workers': {'value': NUM_WORKERS},
        'image_size': {'value': [IMAGE_SIZE, IMAGE_SIZE]},
        'num_classes': {'value': 2},
        'data_dir': {'value': DATA_DIR},
        'output_dir': {'value': OUTPUT_DIR},
        'val_split': {'value': VAL_SPLIT},
        'random_seed': {'value': RANDOM_SEED},
        'use_amp': {'value': True},
        'early_stopping_patience': {'value': 10},
        'scheduler': {'value': 'CosineAnnealingLR'},
        'weight_decay': {'value': 1e-4}
    }
}

print("✅ HPO 搜索空间已配置")
print(f"\n📊 搜索策略: {sweep_config['method']}")
print(f"🎯 优化目标: 最大化 {sweep_config['metric']['name']}")
print(f"🔄 将运行 {HPO_COUNT} 次实验")

---
# 🏋️ 阶段四：定义单次训练函数
## Stage 4: Define Training Function
---

这个函数会被 HPO 系统**自动调用多次**，每次使用不同的超参数。

In [ ]:
# 导入必要的模块
import sys
sys.path.append('/content/drive/MyDrive/FTP')  # 修改为您的项目路径

from src.data.dataset import prepare_dataloaders
from src.data.transforms import get_training_augmentation, get_validation_augmentation
from src.models.model_factory import create_model
from src.models.losses import get_loss_function
from src.train import train_model
from src.utils.utils import set_seed, get_device

print("✅ 训练模块已加载")

In [ ]:
def train_one_run(config=None):
    """
    单次训练运行（被 W&B Sweep 调用）
    
    这个函数会被自动调用多次，每次使用不同的超参数组合
    """
    # 初始化 W&B run
    with wandb.init(config=config) as run:
        config = wandb.config
        
        print("\n" + "="*70)
        print(f"🚀 开始运行: {run.name}")
        print("="*70)
        print("📋 当前配置:")
        for key, value in config.items():
            if not key.startswith('_'):
                print(f"   {key}: {value}")
        print("="*70 + "\n")
        
        # 设置随机种子
        set_seed(config.random_seed)
        
        # 获取设备
        device = get_device()
        
        # 准备数据增强
        train_transform = get_training_augmentation(
            image_size=tuple(config.image_size),
            augmentation_prob=config.augmentation_prob
        )
        val_transform = get_validation_augmentation(
            image_size=tuple(config.image_size)
        )
        
        # 准备数据加载器
        print("📦 准备数据...")
        train_loader, val_loader = prepare_dataloaders(
            data_dir=config.data_dir,
            batch_size=config.batch_size,
            val_split=config.val_split,
            num_workers=config.num_workers,
            random_seed=config.random_seed,
            train_transform=train_transform,
            val_transform=val_transform
        )
        
        # 创建模型
        print(f"\n🏗️  创建模型: {config.model_name} ({config.encoder_name})")
        model = create_model(
            model_name=config.model_name,
            encoder_name=config.encoder_name,
            encoder_weights="imagenet",
            in_channels=3,
            num_classes=config.num_classes
        )
        model = model.to(device)
        
        # 创建损失函数
        criterion = get_loss_function(config.loss_function)
        criterion = criterion.to(device)
        
        # 创建优化器
        if config.optimizer == "Adam":
            optimizer = torch.optim.Adam(
                model.parameters(),
                lr=config.learning_rate,
                weight_decay=config.weight_decay
            )
        elif config.optimizer == "AdamW":
            optimizer = torch.optim.AdamW(
                model.parameters(),
                lr=config.learning_rate,
                weight_decay=config.weight_decay
            )
        
        # 创建学习率调度器
        scheduler = None
        if config.scheduler == "CosineAnnealingLR":
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=config.num_epochs
            )
        
        # 训练模型
        print("\n🏋️  开始训练...\n")
        results = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            device=device,
            num_epochs=config.num_epochs,
            save_dir=config.output_dir,
            use_amp=config.use_amp,
            early_stopping_patience=config.early_stopping_patience,
            log_wandb=True
        )
        
        # 记录最终结果
        print("\n" + "="*70)
        print("✅ 训练完成！")
        print(f"🏆 最佳验证 Dice: {results['best_dice']:.4f}")
        print("="*70 + "\n")
        
        wandb.log({'validation_dice': results['best_dice']})

print("✅ 训练函数已定义")

---
# 🚀 阶段五：启动超参数优化
## Stage 5: Launch Hyperparameter Optimization
---

### ⚡ 这里是魔法发生的地方！

运行下面的单元格后：
1. W&B 会创建一个新的 Sweep（优化任务）
2. Agent 会自动运行 `HPO_COUNT` 次实验
3. 每次实验使用不同的超参数组合
4. 您可以在 W&B 仪表板实时查看进度

**预计时间：** 约 `NUM_EPOCHS × HPO_COUNT × 2` 分钟  
（例如: 30 epochs × 20 runs ≈ 600 分钟 ≈ 10 小时）

**建议：** 先用 `HPO_COUNT=5` 和 `NUM_EPOCHS=10` 进行快速测试！

In [ ]:
# 创建 W&B Sweep
sweep_id = wandb.sweep(
    sweep=sweep_config,
    project=PROJECT_NAME
)

print("\n" + "="*70)
print("🎯 W&B Sweep 已创建！")
print("="*70)
print(f"Sweep ID: {sweep_id}")
print(f"\n🌐 查看实时进度:")
print(f"   https://wandb.ai/{wandb.api.default_entity}/{PROJECT_NAME}/sweeps/{sweep_id}")
print("="*70 + "\n")

In [ ]:
# 启动 W&B Agent（开始自动优化）
print("🤖 启动 W&B Agent...")
print(f"将运行 {HPO_COUNT} 次实验")
print("\n⏰ 提示: 这可能需要几小时，您可以关闭浏览器，稍后回来查看结果")
print("   Colab 会在后台继续运行（注意 Colab 的运行时限制）\n")
print("-"*70 + "\n")

wandb.agent(
    sweep_id=sweep_id,
    function=train_one_run,
    count=HPO_COUNT
)

print("\n" + "="*70)
print("🎉 所有实验完成！")
print("="*70)
print("\n请访问 W&B 仪表板查看结果并选择最佳模型")

---
# 📊 阶段六：分析结果
## Stage 6: Analyze Results
---

### 🔍 如何找到最佳模型？

1. **访问 W&B Sweep 页面**（运行上面单元格后会显示链接）
2. **查看 Parallel Coordinates Plot**：
   - 这个图表会显示哪些参数组合效果最好
   - 高亮的线条代表高 Dice 分数
3. **查看 Parameter Importance**：
   - 显示哪个参数对性能影响最大
4. **选择最佳 Run**：
   - 在表格中找到 `validation_dice` 最高的运行
   - 记录它的配置参数

### 📈 W&B 会自动生成的可视化：
- 🎯 参数重要性分析
- 📉 学习率 vs 性能
- 🌈 平行坐标图
- 📊 性能分布直方图

In [ ]:
# 获取最佳运行的信息
api = wandb.Api()
sweep = api.sweep(f"{PROJECT_NAME}/{sweep_id}")

# 获取所有运行，按 validation_dice 排序
runs = sorted(sweep.runs, key=lambda run: run.summary.get('best_dice', 0), reverse=True)

if runs:
    best_run = runs[0]
    
    print("\n" + "="*70)
    print("🏆 最佳模型配置")
    print("="*70)
    print(f"Run Name: {best_run.name}")
    print(f"Run ID: {best_run.id}")
    print(f"\n🎯 最佳验证 Dice: {best_run.summary.get('best_dice', 'N/A'):.4f}")
    print("\n📋 最佳配置:")
    
    important_params = [
        'model_name', 'encoder_name', 'learning_rate', 'batch_size',
        'loss_function', 'optimizer', 'augmentation_prob'
    ]
    
    for param in important_params:
        value = best_run.config.get(param, 'N/A')
        print(f"   {param}: {value}")
    
    print("\n" + "="*70)
    print(f"\n💡 使用这个配置重新训练更多轮次，可以获得更好的最终模型！")
else:
    print("未找到完成的运行")

---
# 🔮 阶段七：使用最佳模型进行推理
## Stage 7: Inference with Best Model
---

### 📝 TODO:
这部分需要根据您的具体需求实现：
1. 加载最佳模型权重
2. 对完整 WSI 进行滑动窗口推理
3. 生成预测掩码
4. 保存结果用于 QuPath 验证

可以参考 BiaPy 或其他分割项目的推理脚本。

---
# 🎓 总结
## Summary
---

### ✅ 您已完成：

1. ✅ 设置环境并安装所有依赖
2. ✅ 连接 Google Drive 和 W&B
3. ✅ 配置 HPO 搜索空间
4. ✅ 运行自动超参数优化
5. ✅ 分析结果并找到最佳配置

### 🚀 下一步：

1. 使用最佳配置重新训练（更多轮次）
2. 对完整 WSI 进行推理
3. 在 QuPath 中验证结果
4. 根据需要微调参数

### 📚 相关资源：

- [W&B Sweeps 文档](https://docs.wandb.ai/guides/sweeps)
- [Segmentation Models PyTorch](https://github.com/qubvel/segmentation_models.pytorch)
- [项目 GitHub](https://github.com/Zhaokun91/FTP)

---

**祝您训练愉快！** 🎉
